## Atelier Préparation des Données

#### Contexte 
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT.
Chaque capteur 
collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la 
consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de 
fonctionnement et l'état du système de climatisation. 
Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning 
capable de prédire la consommation énergétique ou de détecter les situations anormales. 

Cependant, les données brutes présentent volontairement différents problèmes : valeurs 
manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables 
catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables. 
L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt 
pour le Machine Learning. 

In [1]:
# Importation de librairie
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### <span style="color: #2982fe">Partie 1 – Exploration des données </span>

##### <span style="color: #8eb9f6">1) Chargement des données CSV </span>

In [2]:
df = pd.read_csv("../data/smart_building_raw.csv")

##### <span style="color: #8eb9f6">2) Affichage des premières lignes du dataset</span>

In [3]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


##### <span style="color: #8eb9f6">3) Affichage des dernières lignes du dataset </span>

In [4]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


##### <span style="color: #8eb9f6">4) Nombre d'observations du dataset</span>

In [7]:
print("Nombre de données d'observation : ", df.shape[0])

Nombre de données d'observation :  507


##### <span style="color: #8eb9f6">5) Nombres de variables du dataset</span>

In [9]:
print("Nombre de variables : ", df.shape[1])
print("Nom des variables : ", df.columns.tolist())

Nombre de variables :  14
Nom des variables :  ['id_mesure', 'date', 'batiment', 'type_batiment', 'zone', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


##### <span style="color: #8eb9f6">6) Identifification des variables numériques</span> 

In [15]:
print("Variables de type numérique : ", df.select_dtypes(include=np.number).columns.tolist())

Variables de type numérique :  ['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


##### <span style="color: #8eb9f6">7) Identifification des variables catégorielles</span> 

In [17]:
print('Variables de type catégoriel : ', df.select_dtypes(include='object').columns.tolist())

Variables de type catégoriel :  ['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


##### <span style="color: #8eb9f6">8) Identification des dates</span>

In [30]:
print("Variables date \n: ", df.date)
print("type de la variable date : ", type(df.date[0]))

Variables date 
:  0      2025-02-13 06:00:00
1      2025-03-10 12:00:00
2      2025-05-04 00:00:00
3      2025-01-19 00:00:00
4      2025-04-24 06:00:00
              ...         
502    2025-01-27 12:00:00
503    2025-03-09 12:00:00
504    2025-03-29 00:00:00
505    2025-04-19 18:00:00
506    2025-01-26 12:00:00
Name: date, Length: 507, dtype: object
type de la variable date :  <class 'str'>


##### <span style="color: #8eb9f6">9) Identification des identifiants</span>

In [32]:
print("Les identifiants \n", df.id_mesure)
# print("Les identifiants uniques \n", df.id_mesure.unique())

Les identifiants 
 0      1174
1      1275
2      1493
3      1073
4      1454
       ... 
502    1107
503    1271
504    1349
505    1436
506    1103
Name: id_mesure, Length: 507, dtype: int64


##### <span style="color: #8eb9f6">Informations globales du jeu de données</span>

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_mesure           507 non-null    int64  
 1   date                507 non-null    object 
 2   batiment            507 non-null    object 
 3   type_batiment       503 non-null    object 
 4   zone                507 non-null    object 
 5   temperature         495 non-null    float64
 6   humidite            496 non-null    float64
 7   co2                 500 non-null    float64
 8   occupation          501 non-null    float64
 9   consommation_kwh    502 non-null    float64
 10  mode_climatisation  502 non-null    object 
 11  etat_systeme        507 non-null    object 
 12  jour_semaine        502 non-null    object 
 13  alerte              507 non-null    object 
dtypes: float64(5), int64(1), object(8)
memory usage: 55.6+ KB


##### <span style="color: #8eb9f6">10) Déterminons les statistiques descriptives</span> 

In [35]:
# Statistiques descriptives des variables numériques
df.describe()

,id_mesure,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,24.154141,57.864113,844.150000,44.850299,169.069323
std,144.782769,7.418465,16.026336,582.181386,24.949139,53.164294
min,1001.000000,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,96.000000,160.000000,6000.000000,116.000000,336.200000


In [36]:
# Statistiques descriptives des variables catégorielles
df.describe(include='object')

,date,batiment,type_batiment,zone,mode_climatisation,etat_systeme,jour_semaine,alerte
count,507,507,503,507,502,507,502,507
unique,500,8,16,4,7,3,7,2
top,2025-04-11 06:00:00,B1,Bureau,B,Normal,Normal,Vendredi,Non
freq,2,94,210,143,271,442,75,361
